# 03 · Carga de treino × recuperação

Único notebook que **cruza os dois lados**: o treino (`training_load_daily`) e a fisiologia
(`wellness_daily`). Fica separado justamente porque o cruzamento tem um custo que as análises de um
lado só não têm — ele vive na interseção das janelas, e essa interseção é bem menor que qualquer uma
das duas séries.

**Perguntas deste notebook**

1. Como fitness (CTL), fadiga (ATL) e forma (TSB) evoluíram?
2. A carga de ontem aparece no HRV e no sono de hoje?
3. E o contrário — dormir bem prevê treinar mais no dia seguinte?

Correlação aqui é exploratória: `n` pequeno, sem controle de confundidores e sem correção para testes
múltiplos. Serve para levantar hipótese, não para fechar conclusão.

In [ ]:
from processing.datasets import load_daily
from processing.features import corr_com_pvalor, tabela_correlacoes

import matplotlib.pyplot as plt
import numpy as np

daily = load_daily()

daily[["date", "tss", "ctl", "atl", "tsb", "hrv", "sleep_hours", "sleep_score"]].tail()

## 1. O que `load_daily()` já resolveu

Três decisões estão dentro do loader, e todas mudariam o resultado das correlações se fossem feitas de
outro jeito:

1. **Merge `inner` validado 1:1** — só dias com wellness *e* carga; o `validate` garante que nenhuma
   data se repete e evita duplicação silenciosa de linhas.
2. **Calendário completo** — o intervalo é reindexado dia a dia. Sem isso, um dia sem registro faria
   `shift(1)` puxar o valor de dois ou três dias antes, e "lag1" deixaria de significar ontem.
3. **Lags** — `<coluna>_lag1` carrega o valor de ontem.

A célula abaixo torna isso visível: `atl_lag1` de uma linha é o `atl` da linha anterior.

In [ ]:
print("Janela:", daily["date"].min().date(), "a", daily["date"].max().date(), f"({len(daily)} dias)")
print("Colunas de lag:", [c for c in daily.columns if c.endswith("_lag1")])

daily[["date", "atl", "atl_lag1", "tss", "tss_lag1", "hrv"]].head(6)

## 2. Fitness, fadiga e forma

CTL é a média exponencial longa (fitness acumulado), ATL a curta (fadiga recente), e TSB = CTL − ATL é
a forma. A leitura clássica: ATL acima de CTL por muito tempo é acúmulo de fadiga; TSB positivo por
muito tempo é destreino.

O painel de baixo separa TSB em verde/vermelho só para tornar a travessia do zero legível de relance.

In [ ]:
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(13, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]}
)

ax1.bar(daily["date"], daily["tss"], color="gray", alpha=0.25, width=1, label="TSS do dia")
ax1.plot(daily["date"], daily["ctl"], color="#2980b9", linewidth=2, label="CTL (fitness)")
ax1.plot(daily["date"], daily["atl"], color="#e74c3c", linewidth=1.5, alpha=0.85, label="ATL (fadiga)")
ax1.set_ylabel("TSS / CTL / ATL")
ax1.legend(frameon=False, fontsize=9)
ax1.grid(alpha=0.3)

ax2.fill_between(daily["date"], daily["tsb"], 0, where=daily["tsb"] >= 0, color="#27ae60", alpha=0.5)
ax2.fill_between(daily["date"], daily["tsb"], 0, where=daily["tsb"] < 0, color="#c0392b", alpha=0.5)
ax2.axhline(0, color="black", linewidth=0.8)
ax2.set_ylabel("TSB (forma)")
ax2.grid(alpha=0.3)

ax1.set_title("Carga de treino na janela com dados de wellness")
plt.tight_layout()
plt.show()

## 3. A carga de ontem explica a recuperação de hoje?

Cada linha é um par testado: correlação de Pearson, `n` efetivo (depois de descartar nulos) e p-valor.
A tabela sai ordenada pela força da correlação.

O que se esperaria, se o modelo de carga estiver descrevendo bem este atleta: **ATL de ontem com sinal
negativo** contra HRV de hoje (mais fadiga, menos variabilidade) e **TSB de ontem com sinal positivo**
(mais forma, mais variabilidade). Sinal invertido ou r perto de zero é informação também — quer dizer
que o TSS calculado não está capturando o esforço real, ou que o efeito não aparece em 24 horas.

`significativo` marca p < 0,05 sem correção para múltiplos testes: com ~10 pares testados, um positivo
isolado no limiar é esperado por acaso. Peso mesmo só tem correlação que se repete entre rodadas.

In [ ]:
pares = [
    # carga de ontem -> recuperação de hoje
    ("atl_lag1", "hrv"),
    ("ctl_lag1", "hrv"), # carga de ontem influencia no hrv de hoje (-0,316)
    ("tsb_lag1", "hrv"),
    ("tss_lag1", "hrv"),
    ("tss_lag1", "resting_hr"),
    ("tss_lag1", "sleep_score"),
    ("tss_lag1", "sleep_hours"),
    # mesmo dia (efeito imediato, não predição)
    ("tss", "hrv"),
    # recuperação de ontem -> treino de hoje
    ("hrv_lag1", "tss"),
    ("sleep_score_lag1", "tss"),
    ("sleep_hours_lag1", "tss"),
]

tabela_correlacoes(daily, pares)

## 4. Os dois pares principais, ponto a ponto

Correlação resume em um número o que o scatter mostra por inteiro — inclusive quando o r está sendo
puxado por meia dúzia de dias extremos. Vale olhar a nuvem antes de acreditar no coeficiente.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

paineis = [
    ("atl_lag1", "ATL (fadiga) de ontem", "negativa"),
    ("tsb_lag1", "TSB (forma) de ontem", "positiva"),
]

for ax, (xcol, rotulo, esperado) in zip(axes, paineis):
    sub = daily[[xcol, "hrv"]].dropna()
    ax.scatter(sub[xcol], sub["hrv"], alpha=0.5, s=50, edgecolors="black", linewidth=0.4)

    coef = np.polyfit(sub[xcol], sub["hrv"], 1)
    x_linha = np.linspace(sub[xcol].min(), sub[xcol].max(), 100)
    ax.plot(x_linha, np.polyval(coef, x_linha), "--", color="red", alpha=0.6)

    r = corr_com_pvalor(daily, xcol, "hrv")
    ax.set_title(f"{rotulo}\nr={r['r_pearson']:.2f}, p={r['p_pearson']:.3f}, n={r['n']} (esperado: {esperado})", fontsize=10)
    ax.set_xlabel(rotulo)
    ax.set_ylabel("HRV de hoje (ms)")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

paineis = [
    ("atl_lag1", "ATL (fadiga) de ontem", "negativa"),
    ("tsb_lag1", "TSB (forma) de ontem", "positiva"),
]

for ax, (xcol, rotulo, esperado) in zip(axes, paineis):
    sub = daily[[xcol, "hrv"]].dropna()
    ax.scatter(sub[xcol], sub["hrv"], alpha=0.5, s=50, edgecolors="black", linewidth=0.4)

    coef = np.polyfit(sub[xcol], sub["hrv"], 1)
    x_linha = np.linspace(sub[xcol].min(), sub[xcol].max(), 100)
    ax.plot(x_linha, np.polyval(coef, x_linha), "--", color="red", alpha=0.6)

    r = corr_com_pvalor(daily, xcol, "hrv")
    ax.set_title(f"{rotulo}\nr={r['r_spearman']:.2f}, p={r['p_spearman']:.3f}, n={r['n']} (esperado: {esperado})", fontsize=10)
    ax.set_xlabel(rotulo)
    ax.set_ylabel("HRV de hoje (ms)")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Observações

_A preencher a cada atualização._

- Janela do cruzamento: 119 dias (2026-04-08 a 2026-08-04). Fora dela só existe o lado do treino, e
  nada nesta página vale para o histórico anterior.
- Anotar o sinal e a força de cada correlação a cada rodada. O que interessa não é o r de uma rodada e
  sim se ele **se mantém** quando o `n` cresce.
- Limites conhecidos: sem controle de confundidor (álcool, calor, horário da medição), lag fixo de 1
  dia (o efeito de um bloco forte pode levar 2–3 dias) e TSS estimado a partir de FC, não de potência.